In [12]:
from dotenv import load_dotenv
load_dotenv()


True

In [14]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

loader = Docx2txtLoader('./scenario1.docx')
document_list = loader.load_and_split(text_splitter=text_splitter)

In [16]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings

# 환경변수를 불러옴
load_dotenv()

# Upstage 제공하는 Embedding Model을 활용해서 `chunk`를 vector화
embedding = UpstageEmbeddings(model="solar-embedding-1-large")


In [18]:
from langchain_pinecone import PineconeVectorStore

# 데이터를 처음 저장할 때 
index_name = 'nursing-upsate-index'
database = PineconeVectorStore.from_documents(document_list, embedding, index_name=index_name)


In [4]:
import os

from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from dotenv import load_dotenv

load_dotenv()

index_name = 'nursing-index'
pinecone_api_key = os.environ.get("PINECONE_API_KEY")
host = os.getenv("PINECONE_HOST")

pc = Pinecone(api_key=pinecone_api_key, host=host)


database = PineconeVectorStore.from_documents(document_list, embedding, index_name=index_name)


/Users/jimin/.pyenv/versions/3.11.10/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI()

print(llm.invoke("Hello, how are you?"))

content="Hello! I'm just a computer program, so I don't have feelings, but I'm here and ready to help you with anything you need. How can I assist you today?" additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 13, 'total_tokens': 50, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-BrNywu5P8OKxmzkqGWY49YLBP0duq', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='run--6cb90521-ada5-49d2-b1c5-311b8fd4a5b3-0' usage_metadata={'input_tokens': 13, 'output_tokens': 37, 'total_tokens': 50, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [6]:
query = '1회 용량은 얼마인가요?'

In [7]:
retriever = database.as_retriever(search_kwargs={'k': 4})
retriever.invoke(query)


[Document(id='a6214e2b-e702-4d5a-bbba-6527f1eb9288', metadata={'source': './scenario1.docx'}, page_content='6) 경막외주사 (Epidural)\n\n\n\n\n\nQ3. 1회 용량은 얼마인가요?\n\n    A: 1kg 당 50mg을 하루 8시간 간격으로 투여\n\n     1) 1kg 당  50mg 하루 8시간 동안 투여\n\n     2) 1kg 당 500mg 하루 8시간 간격\n\n     2) 1kg 당 500mg 하루 8시간 동안 투여\n\n     3) 1회 50mg 하루 8시간 간격 투여\n\n     4) 1회 50mg 하루 8시간 동안 투여\n\n     5) 1회 500mg 하루 8시간 간격 투여'),
 Document(id='39b0a223-87be-4516-98bd-4c6b622823a5', metadata={'source': './scenario1.docx'}, page_content='8) Piperacillin 0.5g Vial\n\n    9) Meropenem 0.5g Vial\n\n   10) Cefotaxime 1g Vial\n\n\n\n\n\nQ2. 투약 경로는 무엇인가요?\n\n    A: 정맥투여 (IV)\n\n    1) 근육주사 (IM)\n\n    2) 경구투여 (PO)\n\n    3) 피하주사 (SC)\n\n    4) 흡입 (Inhalation)\n\n    5) 피내주사 (ID)\n\n    6) 경막외주사 (Epidural)\n\n\n\n\n\nQ3. 1회 용량은 얼마인가요?'),
 Document(id='ec6344b0-b455-4215-9480-b0b7d08033d1', metadata={'source': './scenario1.docx'}, page_content='3) 5% 포도당(Dextrose 5%)\n\n  4) 20% 포도당(Dextrose 20%)\n\n\n\nQ12: "희석 시 필요한 용량은?"\n\n  

In [8]:
from langchain import hub

prompt = hub.pull("rlm/rag-prompt")


/Users/jimin/.pyenv/versions/3.11.10/lib/python3.11/site-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [9]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o')


In [10]:
from langchain.chains import RetrievalQA


qa_chain = RetrievalQA.from_chain_type(
    llm, 
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt}
)


In [11]:
ai_message = qa_chain.invoke({"query": query})
print(ai_message
)

{'query': '1회 용량은 얼마인가요?', 'result': '1회 용량은 1회 500mg 하루 8시간 간격으로 투여합니다.'}
